In [1]:
import json
import weaviate
from weaviate.classes.config import Configure, VectorDistances, Property, DataType
import numpy as np

In [2]:
CHILDREN_JSONL = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_children.jsonl"
EMBEDDINGS_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/child_embeddings.npy"
ID_MAP_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/child_id_map.jsonl"
COLLECTION_NAME = "AlzheimersChunk"

In [3]:
client = weaviate.connect_to_local()

In [4]:
if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

In [5]:
collection = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=None,
    vector_index_config=Configure.VectorIndex.hnsw(
        distance_metric=VectorDistances.DOT
    ),
    properties=[
        Property(name="chunk_id", data_type=DataType.TEXT),
        Property(name="parent_id", data_type=DataType.TEXT),
        Property(name="doc_id", data_type=DataType.TEXT),
        Property(name="doc_title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT),
        Property(name="section_type", data_type=DataType.TEXT),
        Property(name="heading_path", data_type=DataType.TEXT_ARRAY),
        Property(name="kwd", data_type=DataType.TEXT_ARRAY),
        Property(name="is_atomic", data_type=DataType.BOOL),
        Property(name="token_count", data_type=DataType.NUMBER),
    ]
)

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/weaviate/warnings.py:206: DeprecationWarning: Dep025: You are using the `vector_index_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead defining `vector_index_config` as a sub-argument.
            
  warnings.warn(


In [6]:
# --- Load data ---
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

children = load_jsonl(CHILDREN_JSONL)
id_map = load_jsonl(ID_MAP_PATH)
embeddings = np.load(EMBEDDINGS_PATH)

assert len(children) == len(id_map) == embeddings.shape[0], \
    "Mismatch between children, id_map, and embeddings — do not proceed until this is fixed"

In [7]:
with collection.batch.dynamic() as batch:
    for i, chunk in enumerate(children):
        properties = {
            "chunk_id": chunk["chunk_id"],
            "parent_id": chunk.get("parent_id"),
            "doc_id": chunk.get("doc_id"),
            "doc_title": chunk.get("doc_title", ""),
            "text": chunk["text"],
            "section_type": chunk.get("section_type", ""),
            "heading_path": chunk.get("heading_path", "").split(" > ") if isinstance(chunk.get("heading_path"), str) else chunk.get("heading_path", []),
            "kwd": chunk.get("kwd", []),
            "is_atomic": chunk.get("is_atomic", False),
            "token_count": chunk.get("token_count", 0),
        }
        batch.add_object(
            properties=properties,
            vector=embeddings[i].tolist(),
        )

        if batch.number_errors > 0:
            print(f"Errors so far: {batch.number_errors}")

failed = collection.batch.failed_objects
if failed:
    print(f"{len(failed)} objects failed to import — inspect before trusting the index")
else:
    print(f"Imported {len(children)} chunks successfully")

client.close()

Imported 56368 chunks successfully
